In [3]:
#!/usr/bin/env python3
import os
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, GenerationConfig
from optimum.onnxruntime import ORTModelForCausalLM

ONNX_REPO = "manu02/gemma-3-1b-it-4bit-lora-dpo-aligned-onnx"
LOCAL_DIR = Path("./hf_onnx_model")
PROMPT = "Explain quantum computing in simple terms."

# 1) Download repo snapshot so config.json + model.onnx + model.onnx.data live together
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=ONNX_REPO,
    local_dir=str(LOCAL_DIR),
    local_dir_use_symlinks=False,
)

# 2) Tokenizer: prefer tokenizer inside ONNX repo; fallback to original transformers repo if needed
try:
    tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_DIR), use_fast=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(
        "manu02/gemma-3-1b-it-4bit-lora-dpo-aligned",
        use_fast=True,
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3) Load ORT model with cache disabled (your ONNX does not support pkv reuse)
model = ORTModelForCausalLM.from_pretrained(
    str(LOCAL_DIR),
    file_name="model.onnx",
    provider="CPUExecutionProvider",
    use_cache=False,  # <-- FIX
)

# Also force the config + generation_config to not use cache
model.config.use_cache = False
gen_cfg = GenerationConfig.from_model_config(model.config)
gen_cfg.use_cache = False

# 4) Generate (no cache)
inputs = tokenizer(PROMPT, return_tensors="pt")

with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        generation_config=gen_cfg,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(out_ids[0], skip_special_tokens=True))


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00,  6.87it/s]
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Explain quantum computing in simple terms.

Okay, let's break down quantum computing. It's a fundamentally different way of computing than the computers we use every day. Here's a simplified explanation:

**1. Classical Computers: Bits and 0s and 1s**

*   Think of a regular computer like a light switch. It can be either on (representing 1) or off (representing 0).
*   Every piece of information a computer stores and processes is represented by these bits.

**2. Quantum Computers: Qubits**

*   Quantum computers use *qubits


In [6]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
import os

# Download ONNX repo locally
onnx_dir = snapshot_download(repo_id="manu02/gemma-3-1b-it-4bit-lora-dpo-aligned-onnx")

# Find model.onnx (handles external data files)
onnx_path = os.path.join(onnx_dir, "model.onnx")

# Load tokenizer (fallback to base repo if needed)
try:
    tokenizer = AutoTokenizer.from_pretrained(onnx_dir)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained("manu02/gemma-3-1b-it-4bit-lora-dpo-aligned")

from optimum.onnxruntime import ORTModelForCausalLM
from transformers import GenerationConfig

# Load model with cache disabled (required for ONNX)
model = ORTModelForCausalLM.from_pretrained(
    onnx_dir,
    file_name="model.onnx",
    provider="CPUExecutionProvider",
    use_cache=False,
)
model.config.use_cache = False
gen_cfg = GenerationConfig.from_model_config(model.config)
gen_cfg.use_cache = False

inputs = tokenizer("Hello, world!", return_tensors="pt")
outputs = model.generate(
    **inputs,
    generation_config=gen_cfg,
    max_new_tokens=32,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(outputs[0]))

# Fallback: pure onnxruntime
import onnxruntime as ort
import numpy as np
inputs = tokenizer("Hello, world!", return_tensors="np")
session = ort.InferenceSession(onnx_path)
input_feed = dict(inputs)
outputs = session.run(None, input_feed)
print(outputs)


Fetching 5 files: 100%|██████████| 5/5 [02:23<00:00, 28.63s/it]


<bos>Hello, world!

I'm a large language model, and I'm here to help you with your requests. I can:

*   **Answer questions:** I
[array([[[-14.226649  ,  -1.1276453 ,   4.660422  , ..., -14.709715  ,
         -14.754735  , -14.670811  ],
        [-13.768501  ,   1.4250515 ,   3.1874652 , ..., -15.133059  ,
         -14.92111   , -14.954026  ],
        [-15.419237  ,   0.53714275,  -2.6596184 , ..., -16.118906  ,
         -15.994941  , -16.099594  ],
        [-16.02862   ,  -6.111081  ,   0.89784443, ..., -17.480232  ,
         -17.520792  , -17.36332   ],
        [-13.747504  ,  -3.4913127 ,  -2.0501058 , ..., -14.2555    ,
         -14.251692  , -14.254672  ]]],
      shape=(1, 5, 262144), dtype=float32)]
